This notebook extracts information about senses and synonyms from WordNet

In [1]:
import os
import pandas as pd
import pickle
from nltk.corpus import wordnet as wn

Print version numbers for reproducibility

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-09-25T10:03:09.263883+10:00

Python implementation: CPython
Python version       : 3.11.5
IPython version      : 8.12.3

Compiler    : MSC v.1936 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 154 Stepping 4, GenuineIntel
CPU cores   : 12
Architecture: 64bit

pandas: 2.2.3



In [3]:
project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

file_path = os.path.join(project_folder, "data", "wordnet_mapping_manual.csv")

lemmas = pd.read_csv(file_path, encoding='utf-8')
lemmas['final_synset'] = lemmas['synset_assignment'].fillna(lemmas['pwn_synset'])

lemmas.head()

,concepticon_id,name,description,pwn_synset,wordnet_gloss,wordnet_definition,synset_validation,synset_assignment,comment,final_synset
0,965,world,The Earth with all its inhabitants and all thi...,NaN,NaN,NaN,NaN,n9270894,NaN,n9270894
1,626,land,A specified geographical tract of the Earth's ...,n09334396,land//dry land//earth//ground//solid ground//t...,"the solid part of the earth's surface; ""the pl...",1.0,NaN,NaN,n09334396
2,1228,earth (soil),The soft and loose material forming a great pa...,n09335240,land//ground//soil,material in the top layer of the surface of th...,1.0,NaN,NaN,n09335240
3,2,dust,Any kind of solid material divided in particle...,n14839846,dust,fine powdery material such as dry earth or pol...,1.0,NaN,NaN,n14839846
4,640,mud,A mixture of clay and/or silt with water to fo...,n14956325,mud//clay,water soaked soil; soft wet earth,1.0,NaN,NaN,n14956325


We create a list of synonyms based on wordnet ID.

In [4]:
def get_synonyms_from_synset(final_synset):
    if not isinstance(final_synset, str):
        return []

    if len(final_synset) < 2 or not final_synset[1:].isdigit():
        return []

    try:
        pos = final_synset[0]
        offset = int(final_synset[1:])
        
        synset = wn.synset_from_pos_and_offset(pos, offset)
        
        synonyms = {lemma.name() for lemma in synset.lemmas()}
        return list(synonyms)
    except Exception as e:
        return []

lemmas['synonym'] = lemmas['final_synset'].apply(get_synonyms_from_synset)
synonyms = lemmas.explode('synonym').reset_index(drop=True)
synonyms.head()

,concepticon_id,name,description,pwn_synset,wordnet_gloss,wordnet_definition,synset_validation,synset_assignment,comment,final_synset,synonym
0,965,world,The Earth with all its inhabitants and all thi...,NaN,NaN,NaN,NaN,n9270894,NaN,n9270894,Earth
1,965,world,The Earth with all its inhabitants and all thi...,NaN,NaN,NaN,NaN,n9270894,NaN,n9270894,world
2,965,world,The Earth with all its inhabitants and all thi...,NaN,NaN,NaN,NaN,n9270894,NaN,n9270894,globe
3,965,world,The Earth with all its inhabitants and all thi...,NaN,NaN,NaN,NaN,n9270894,NaN,n9270894,earth
4,626,land,A specified geographical tract of the Earth's ...,n09334396,land//dry land//earth//ground//solid ground//t...,"the solid part of the earth's surface; ""the pl...",1.0,NaN,NaN,n09334396,land


For these synonymous words, we add the most regular sense from word sense disambiguation data.

In [5]:
pkl_path = os.path.join(project_folder, "rawdata", "wsd_coha.pkl")

with open(pkl_path, "rb") as f:
    data = pickle.load(f)

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys([1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000])


We will use the word sense disambiguation data based on COHA data between 2000-2009.

In [6]:
df = pd.DataFrame.from_dict(data[2000], orient="index", columns=["count"])
df.index = pd.MultiIndex.from_tuples(df.index, names=["lemma", "wordnet_id"])
df = df.reset_index()
df.head()

,lemma,wordnet_id,count
0,pause,wn:15271008n,733
1,so,wn:00147272r,1031
2,silence,wn:13925550n,179
3,call,wn:00789448v,5135
4,dissertation,wn:06409085n,100


In [7]:
def get_definition(wordnet_id):
    raw_id = wordnet_id[3:]
    
    sense_id = raw_id[:-1] 
    pos = raw_id[-1]      
    
    synset = wn.synset_from_pos_and_offset(pos, int(sense_id))
    return pd.Series([synset.definition()], index=["definition"])

df[["definition"]] = df["wordnet_id"].apply(get_definition)
df.head()

,lemma,wordnet_id,count,definition
0,pause,wn:15271008n,733,a time interval during which there is a tempor...
1,so,wn:00147272r,1031,in the same way; also
2,silence,wn:13925550n,179,the state of being silent (as when no one is s...
3,call,wn:00789448v,5135,get or try to get into communication (with som...
4,dissertation,wn:06409085n,100,a treatise advancing a new point of view resul...


In [8]:
df_sorted = df.sort_values(by=['lemma', 'count'], ascending=[True, False])
df_most_frequent = df_sorted.drop_duplicates(subset='lemma', keep='first')

def reformat_wordnet_id(wordnet_id):
    pos = wordnet_id[-1]
    offset = wordnet_id[3:-1]
    return f"{pos}{offset}"

df_most_frequent.loc[:, 'wordnet_id'] = df_most_frequent['wordnet_id'].apply(reformat_wordnet_id)
wsd = df_most_frequent.rename(columns={'lemma': 'synonym'})
wsd.head(10)

,synonym,wordnet_id,count,definition
64657,.22,n04502851,1,a .22 caliber firearm (pistol or rifle)
43352,1,n13742573,3,the smallest whole number or a numeral represe...
21636,10,n13746512,5,the cardinal number that is the sum of nine an...
37299,100th,s02209423,26,the ordinal number of one hundred in counting ...
50171,101st,s02209551,9,the ordinal number of one hundred one in count...
60072,105th,s02209678,3,the ordinal number of one hundred five in coun...
28640,10th,s02203373,116,coming next after the ninth and just before th...
69101,110th,s02209806,2,the ordinal number of one hundred ten in count...
57038,115th,s02209933,2,the ordinal number of one hundred fifteen in c...
38605,11th,s02203500,75,coming next after the tenth and just before th...


In [9]:
combined = pd.merge(synonyms, wsd, on='synonym', how='left')


# Ensure 'synonym' column is a string type and handle NaN values
combined['synonym'] = combined['synonym'].fillna('').astype(str)

# Filter out compounds (synonyms that contain underscores)
combined_filtered = combined[~combined['synonym'].str.contains('_')].copy()

# Create the 'same_synset' column based on the condition using .loc
combined_filtered.loc[:, 'same_synset'] = combined_filtered.apply(
    lambda row: 'same' if row['final_synset'] == row['wordnet_id'] else 'different', axis=1
)

# Select the relevant columns
combined_filtered = combined_filtered[['concepticon_id', 'name', 'description', 'final_synset', 
                                       'synonym', 'wordnet_id', 'count', 'definition', 'same_synset']]

combined_filtered = combined_filtered.sort_values(by=['concepticon_id', 'name', 'synonym'])

# Write the filtered dataframe to synonym_mapping.csv
synonym_mapping_file = os.path.join(project_folder, "data", "synonym_mapping.csv")
combined_filtered.to_csv(synonym_mapping_file, index=False, encoding='utf-8')
